# LSTM Networks

Run the cells from top to bottom to verify a scalar gated-memory calculation, compare a manual transition with PyTorch, and inspect a tiny sequence classifier. All data are generated locally and the examples are for mechanics, not benchmark claims.

In [1]:
import torch
from torch import nn

torch.manual_seed(23)
torch.set_num_threads(1)
dtype = torch.float64
print('CPU | LSTM gate arithmetic and PyTorch checks')

CPU | LSTM gate arithmetic and PyTorch checks


## Check a scalar LSTM transition

Use the note's prior cell and gate pre-activations. This makes the arithmetic inspectable before using a built-in recurrent layer.

In [3]:
sigmoid = torch.sigmoid
previous_cell = torch.tensor(0.5, dtype=dtype)
forget = sigmoid(torch.tensor(0.0, dtype=dtype))
input_gate = sigmoid(torch.tensor(0.0, dtype=dtype))
candidate = torch.tanh(torch.tensor(1.0, dtype=dtype))
output_gate = sigmoid(torch.tensor(0.0, dtype=dtype))
cell = forget * previous_cell + input_gate * candidate
hidden = output_gate * torch.tanh(cell)
torch.testing.assert_close(cell, torch.tensor(0.6307970779778824, dtype=dtype))
torch.testing.assert_close(hidden, torch.tensor(0.27930041077890644, dtype=dtype))
print(f'f={forget.item():.3f}, i={input_gate.item():.3f}, candidate={candidate.item():.4f}, o={output_gate.item():.3f}')
print(f'cell={cell.item():.6f}, hidden={hidden.item():.6f}')

f=0.500, i=0.500, candidate=0.7616, o=0.500
cell=0.630797, hidden=0.279300


## Inspect a built-in LSTM

PyTorch returns the output sequence plus the final hidden and cell states. The layer stores gate parameters internally; the scalar calculation above exposes the underlying update.

In [ ]:
layer = nn.LSTM(input_size=1, hidden_size=3, batch_first=True).double()
sequence = torch.tensor([[[0.2], [0.7], [-0.1], [0.4]]], dtype=dtype)
outputs, (final_hidden, final_cell) = layer(sequence)
assert outputs.shape == (1, 4, 3)
assert final_hidden.shape == final_cell.shape == (1, 1, 3)
torch.testing.assert_close(outputs[:, -1], final_hidden[-1])
print('output sequence:', tuple(outputs.shape))
print('final hidden/cell:', tuple(final_hidden.shape), tuple(final_cell.shape))

## Train a small order classifier

The label is whether a marker occurs before or after a distractor. Counts alone cannot distinguish the order. This deliberately tiny synthetic task illustrates the input/output interface, not general sequence performance.

In [4]:
def make_order_data(count=240, length=6):
    features, labels = [], []
    for index in range(count):
        marker, distractor = (1, 3) if index % 2 == 0 else (3, 1)
        row = torch.zeros(length, 2)
        row[marker, 0] = 1
        row[distractor, 1] = 1
        features.append(row)
        labels.append(index % 2)
    return torch.stack(features), torch.tensor(labels)

features, labels = make_order_data()
class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(2, 8, batch_first=True)
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        sequence_output, _ = self.rnn(x)
        return self.head(sequence_output[:, -1])

classifier = LSTMClassifier()
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.03)
for _ in range(80):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(classifier(features), labels)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    accuracy = (classifier(features).argmax(1) == labels).float().mean().item()
assert accuracy > 0.95
print(f'train loss={loss.item():.4f} | toy accuracy={accuracy:.1%}')

train loss=0.0001 | toy accuracy=100.0%


The data above are repeated fixed templates, so this accuracy is only a smoke test that the model can fit a simple order distinction. For real sequence tasks, separate examples by source/time, use validation and test sets, and handle padding and sequence lengths explicitly.